# Evaluating Advanced Machine Learning Models for Financial Risk Prediction using FinBench

**Objective:** Compare traditional machine learning models (Logistic Regression, Random Forest, Gradient Boosting, Neural Network) against a FinPT-like approach — converting tabular data into natural-language customer profiles and embedding them with a sentence transformer — on the [FinBench](https://huggingface.co/datasets/yuweiyin/FinBench) benchmark for financial risk prediction.

Based on the FinPT paper: Yin et al., 2023, ["FinPT: Financial Risk Prediction with Profile Tuning on Pretrained Foundation Models"](https://arxiv.org/abs/2308.00065).

We evaluate on two FinBench tasks:
- **cd2** — Credit Default prediction
- **ld2** — Loan Default prediction

This notebook runs both end to end: data loading, baseline training, the FinPT-like pipeline, and a side-by-side comparison.

## Setup

In [ ]:
!pip install -q "datasets==3.6.0" sentence-transformers scikit-learn pandas matplotlib

In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from data_utils import load_finbench_split
from baseline_models import train_and_evaluate_baselines
from finpt_model import train_and_evaluate_finpt_like

## 1. Data Collection and Preprocessing

FinBench tasks are fetched directly from the HuggingFace Hub. Each task ships with metadata describing which columns are numerical vs. categorical, so we don't need to hand-infer dtypes.

We keep an untouched copy of the raw feature values (`X_raw`) — these are used for two different purposes downstream:
- **Baseline models:** numerical columns get `StandardScaler`'d and categorical columns get one-hot encoded, inside an sklearn `Pipeline` (avoids leakage since the scaler/encoder are fit only on the training split).
- **FinPT-like model:** the *raw* values are converted into natural-language sentences (e.g. `"income is 45000, loan_grade is 3, ..."`). Using raw values here — rather than already-standardized z-scores — matters: a sentence like `"income is -0.27"` is not actually natural language a pretrained encoder has useful priors over, whereas `"income is 45000"` is.

In [ ]:
DATASET_NAME = "cd2"  # change to "ld2" to run the Loan Default task instead

data = load_finbench_split(DATASET_NAME, split="train")

print("Numerical columns:", data.numerical_cols)
print("Categorical columns:", data.categorical_cols)
print("Shape:", data.X_raw.shape)
data.X_raw.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data.X_raw, data.y, test_size=0.2, random_state=42
)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

## 2. Baseline Model Training

Logistic Regression, Random Forest, Gradient Boosting, and a Neural Network (MLP) are each wrapped in a preprocessing pipeline and evaluated on accuracy, precision, recall, F1-score, and AUC-ROC.

In [ ]:
baseline_results = train_and_evaluate_baselines(
    X_train, y_train, X_test, y_test,
    data.numerical_cols, data.categorical_cols
)
baseline_results

## 3. FinPT-like Model

We convert each row into a natural-language customer profile, embed it with [`sentence-transformers/all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2), and train a logistic regression classifier on the resulting 384-dim embeddings — a lightweight stand-in for FinPT's full profile-tuning of a pretrained foundation model.

In [ ]:
finpt_metrics = train_and_evaluate_finpt_like(X_train, y_train, X_test, y_test)
finpt_metrics

## 4. Model Comparison

In [ ]:
combined = baseline_results.copy()
combined.loc["FinPT-like Model"] = finpt_metrics

print(f"Combined Model Performance — '{DATASET_NAME}' dataset:")
combined

In [ ]:
plt.figure(figsize=(12, 6))
combined[["Accuracy", "Precision", "Recall", "F1-score"]].plot(
    kind="bar", figsize=(12, 6), edgecolor="black"
)
plt.title(f"Model Performance Comparison — '{DATASET_NAME}' (Baseline vs. FinPT-like)")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=45)
plt.legend(title="Metrics")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.savefig(f"../results/{DATASET_NAME}_model_performance.png")
plt.show()

## 5. Run Both Tasks Back-to-Back (optional)

Instead of manually changing `DATASET_NAME` above and re-running the whole notebook (which is how the original version of this project worked, and how a copy/paste error crept into the final report's results table), this loops over both tasks and saves results separately so cd2 and ld2 numbers can never get mixed up.

In [ ]:
import os
os.makedirs("../results", exist_ok=True)

all_results = {}
for ds_name in ["cd2", "ld2"]:
    d = load_finbench_split(ds_name, split="train")
    Xtr, Xte, ytr, yte = train_test_split(d.X_raw, d.y, test_size=0.2, random_state=42)

    base = train_and_evaluate_baselines(Xtr, ytr, Xte, yte, d.numerical_cols, d.categorical_cols, verbose=False)
    fpt = train_and_evaluate_finpt_like(Xtr, ytr, Xte, yte, verbose=False)

    res = base.copy()
    res.loc["FinPT-like Model"] = fpt
    res.to_csv(f"../results/{ds_name}_model_performance.csv")
    all_results[ds_name] = res
    print(f"\n--- {ds_name} ---")
    print(res.round(3))

## 6. Conclusion

**Findings:** Baseline tree-based and linear models (Random Forest, Gradient Boosting, Logistic Regression) trained on properly preprocessed tabular features remain strong, fast, and easy-to-interpret baselines for structured financial risk data. The FinPT-like approach — converting rows to text and embedding with a general-purpose sentence transformer — under-performed the baselines on both tasks in this experiment, most notably on recall for the minority (risky/default) class.

**Why the FinPT-like model lagged here:**
- `all-MiniLM-L6-v2` is a general-purpose sentence encoder, not fine-tuned on financial profile text or the downstream classification objective. The original FinPT paper profile-tunes the *foundation model itself* on labeled risk data; here we only train a lightweight classifier on top of frozen embeddings, which is a much weaker setup.
- Tabular columns turned into flat "field is value" sentences lose the relational/structural information (e.g., scale, ordering, feature importance) that tree-based models exploit directly.
- Class imbalance hits the embedding-based classifier harder — the minority class signal is diluted across the embedding's 384 dimensions rather than isolated in a few informative columns.

**Strengths/weaknesses considered:**
- *Interpretability:* Logistic Regression and tree-based models offer feature importances / coefficients; the FinPT-like model's embeddings are opaque without additional explainability tooling.
- *Training time:* Baselines train in seconds; the FinPT-like pipeline is bottlenecked by encoding every row through a transformer.
- *Generalization:* With more labeled data and full profile-tuning (rather than frozen embeddings + linear probe), the gap to baselines would likely narrow — this is consistent with what the FinPT paper itself reports for low-data tabular settings.

**Future work:** profile-tune (fine-tune) the sentence encoder or a small LLM directly on the classification objective rather than freezing it; combine structured tabular features with the text embedding in a hybrid model; and pretrain or adapt the encoder on financial-domain text before applying it to FinBench profiles.